# Vehicle Audio Classification Pipeline

**7 Classes:** Background, Bicycle, Bus, Car, Motorcycle, Tram, Truck

Processes WAV files directly → Mel-Spectrograms → CNN Classification

---

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import os
import librosa
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Configuration

In [ ]:
# ============================================================
# PATHS — update these for your environment (local or Colab)
# ============================================================
DATASET_DIR = os.path.normpath(r"/content/drive/MyDrive/IOT/DataSet_By_Type")
MODEL_DIR = os.path.normpath(r"/content/drive/MyDrive/IOT/Trained Models")
OUTPUT_DIR = os.path.normpath(r"/content/drive/MyDrive/IOT/")
CACHE_FILE = os.path.normpath(os.path.join(OUTPUT_DIR, "audio_spectrograms_cache.npz"))

# ============================================================
# AUDIO / SPECTROGRAM PARAMETERS
# ============================================================
N_FFT = 512
HOP_LENGTH = 64
N_MELS = 64
# No resizing needed — all audio clips are 2s, so spectrograms have uniform dimensions

# ============================================================
# TRAINING PARAMETERS
# ============================================================
BATCH_SIZE = 128
EPOCHS = 30
INITIAL_LR = 0.0001
RANDOM_STATE = 42

# ============================================================
# CLASS DEFINITIONS
# ============================================================
CLASS_NAMES = ['Background', 'Bicycle', 'Bus', 'Car', 'Motorcycle', 'Tram', 'Truck']
CLASS_LABEL_MAP = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

print(f"Class Labels: {CLASS_LABEL_MAP}")
print(f"Number of Classes: {NUM_CLASSES}")

## 3. Audio Processing Functions

In [ ]:
def wav_to_mel_spectrogram(audio_path):
    """
    Convert a WAV file to a Mel-Spectrogram array (no resizing).

    Args:
        audio_path: Path to the WAV file.

    Returns:
        2D numpy array of shape (N_MELS, time_steps), or None on failure.
    """
    try:
        # Load audio (first channel if stereo)
        audio, sr = librosa.load(audio_path, sr=None, mono=True)

        # Skip very short or silent files
        if len(audio) < N_FFT or np.max(np.abs(audio)) == 0:
            return None

        # Normalize
        audio = audio / np.max(np.abs(audio))

        # Compute Mel-Spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=audio, sr=sr,
            n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
        )

        # Convert to dB
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Normalize to [0, 1]
        mel_min = mel_spec_db.min()
        mel_max = mel_spec_db.max()
        if mel_max - mel_min > 0:
            mel_spec_db = (mel_spec_db - mel_min) / (mel_max - mel_min)
        else:
            mel_spec_db = np.zeros_like(mel_spec_db)

        return mel_spec_db.astype(np.float32)

    except Exception as e:
        print(f"\n[ERROR] Failed to process {os.path.basename(audio_path)}: {e}")
        return None

## 4. Data Loading (with NPZ Caching)

In [ ]:
def load_dataset():
    """
    Load all WAV files from DataSet_By_Type, convert to spectrograms.
    Uses an .npz cache file for fast re-runs.

    Returns:
        spectrograms: numpy array (N, n_mels, time_steps)
        labels: numpy array (N,)
        file_names: list of str
    """
    # Check for cache
    if os.path.exists(CACHE_FILE):
        print(f"\n[CACHE] Loading cached spectrograms from: {CACHE_FILE}")
        data = np.load(CACHE_FILE, allow_pickle=True)
        return data['spectrograms'], data['labels'], list(data['file_names'])

    print(f"\n[LOAD] Processing WAV files from: {DATASET_DIR}")
    total_files = sum(
        len(os.listdir(os.path.join(DATASET_DIR, c)))
        for c in CLASS_NAMES
        if os.path.isdir(os.path.join(DATASET_DIR, c))
    )
    print(f"[INFO] This may take several minutes for {total_files} files...\n")

    spectrograms = []
    labels = []
    file_names = []

    for class_name in CLASS_NAMES:
        class_dir = os.path.join(DATASET_DIR, class_name)
        if not os.path.isdir(class_dir):
            print(f"[WARN] Directory not found: {class_dir}")
            continue

        label = CLASS_LABEL_MAP[class_name]
        wav_files = [f for f in os.listdir(class_dir) if f.lower().endswith('.wav')]

        print(f"Processing {class_name} ({len(wav_files)} files)...")
        success = 0
        failed = 0

        for wav_file in tqdm(wav_files, desc=f"  {class_name}", leave=True):
            wav_path = os.path.join(class_dir, wav_file)
            spec = wav_to_mel_spectrogram(wav_path)

            if spec is not None:
                spectrograms.append(spec)
                labels.append(label)
                file_names.append(wav_file)
                success += 1
            else:
                failed += 1

        print(f"  → {success} OK, {failed} failed\n")

    spectrograms = np.array(spectrograms, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)

    # Save cache
    print(f"[CACHE] Saving cache to: {CACHE_FILE}")
    np.savez_compressed(CACHE_FILE, spectrograms=spectrograms, labels=labels, file_names=file_names)

    print(f"\n[DONE] Total spectrograms: {len(spectrograms)}")
    return spectrograms, labels, file_names

In [ ]:
spectrograms, labels, file_names = load_dataset()

# Add channel dimension: (N, n_mels, time_steps) → (N, n_mels, time_steps, 1)
spectrograms = spectrograms[..., np.newaxis]

# Shuffle
spectrograms, labels, file_names = shuffle(spectrograms, labels, file_names, random_state=RANDOM_STATE)

print(f"\nSpectrograms shape: {spectrograms.shape}")
print(f"Labels shape: {labels.shape}")

### Visualize Sample Spectrograms

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(21, 3))
for i, class_name in enumerate(CLASS_NAMES):
    idx = np.where(labels == i)[0][0]
    axes[i].imshow(spectrograms[idx, :, :, 0], aspect='auto', origin='lower', cmap='magma')
    axes[i].set_title(class_name, fontsize=10)
    axes[i].axis('off')
plt.suptitle('Sample Mel-Spectrograms (one per class)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Train / Validation / Test Split (72 / 8 / 20)

In [ ]:
def display_class_distribution(labels_array, dataset_type):
    """Print per-class sample counts."""
    counts = np.bincount(labels_array, minlength=NUM_CLASSES)
    print(f"\n{dataset_type} Dataset Distribution:")
    for i, count in enumerate(counts):
        pct = (count / len(labels_array)) * 100
        print(f"  {CLASS_NAMES[i]:<15}: {count:>5} samples ({pct:>5.1f}%)")
    print(f"  {'Total':<15}: {len(labels_array):>5} samples")

In [ ]:
X_trainval, X_test, y_trainval, y_test, fn_trainval, fn_test = train_test_split(
    spectrograms, labels, file_names,
    test_size=0.2, random_state=RANDOM_STATE, stratify=labels
)

X_train, X_val, y_train, y_val, fn_train, fn_val = train_test_split(
    X_trainval, y_trainval, fn_trainval,
    test_size=0.1, random_state=RANDOM_STATE, stratify=y_trainval
)

display_class_distribution(y_train, 'Training')
display_class_distribution(y_val, 'Validation')
display_class_distribution(y_test, 'Test')

# One-hot encode
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat = to_categorical(y_val, NUM_CLASSES)
y_test_cat = to_categorical(y_test, NUM_CLASSES)

### Class Distribution Bar Chart

In [ ]:
train_counts = np.bincount(y_train, minlength=NUM_CLASSES)
val_counts = np.bincount(y_val, minlength=NUM_CLASSES)
test_counts = np.bincount(y_test, minlength=NUM_CLASSES)

x = np.arange(NUM_CLASSES)
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, train_counts, width, label='Train', color='#4C72B0')
ax.bar(x, val_counts, width, label='Validation', color='#55A868')
ax.bar(x + width, test_counts, width, label='Test', color='#C44E52')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_ylabel('Number of Samples')
ax.set_title('Class Distribution across Splits')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Compute Class Weights (Handle Imbalance)

In [ ]:
weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weight_dict = {i: w for i, w in enumerate(weights)}

print("Computed class weights:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:<15}: weight = {class_weight_dict[i]:.4f}")

## 7. Build CNN Model

In [ ]:
def build_model(input_shape):
    """
    Build a CNN for multi-class spectrogram classification.
    Input shape: (n_mels, time_steps, 1) — full-resolution Mel-Spectrogram.
    """
    model = tf.keras.Sequential([
        # Block 1
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                               input_shape=input_shape),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.25),

        # Block 2
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.25),

        # Block 3
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.3),

        # Block 4
        tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.3),

        # Classifier head — GlobalAveragePooling works with any spatial size
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    return model

input_shape = X_train.shape[1:]  # (n_mels, time_steps, 1)
print(f"Input shape: {input_shape}")
model = build_model(input_shape)
model.summary()

## 8. Compile and Train

In [ ]:
def step_decay(epoch):
    """Reduce learning rate by 10x every 15 epochs."""
    drop = 0.1
    epochs_drop = 15
    lr = INITIAL_LR * (drop ** (epoch // epochs_drop))
    return lr

optimizer = tf.keras.optimizers.Adam(learning_rate=INITIAL_LR)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.LearningRateScheduler(step_decay, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    )
]

history = model.fit(
    X_train, y_train_cat,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weight_dict,
    callbacks=callbacks
)

## 9. Save Model

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, "vehicle_audio_classifier.h5")
model.save(model_path)
print(f"Model saved to: {model_path}")

## 10. Training History Plots

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axs[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axs[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axs[0].set_title('Model Accuracy', fontsize=14)
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Accuracy')
axs[0].legend()
axs[0].grid(True, alpha=0.3)

# Loss
axs[1].plot(history.history['loss'], label='Train', linewidth=2)
axs[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axs[1].set_title('Model Loss', fontsize=14)
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Loss')
axs[1].legend()
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'training_history_audio.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Training history saved to: {save_path}")
plt.show()

## 11. Evaluate on Test Set

In [ ]:
predictions = model.predict(X_test)
pred_classes = np.argmax(predictions, axis=1)
confidence = np.max(predictions, axis=1)

# Results DataFrame
results_df = pd.DataFrame({
    'file_name': fn_test,
    'actual_label': [CLASS_NAMES[l] for l in y_test],
    'predicted_label': [CLASS_NAMES[p] for p in pred_classes],
    'confidence': confidence,
    'correct': y_test == pred_classes
})

results_csv = os.path.join(OUTPUT_DIR, 'classification_results_audio.csv')
results_df.to_csv(results_csv, index=False)
print(f"Results saved to: {results_csv}")

correct = results_df['correct'].sum()
total = len(results_df)
acc = correct / total * 100
print(f"\nTest Accuracy: {correct}/{total} ({acc:.2f}%)")

## 12. Classification Report

In [ ]:
print(classification_report(y_test, pred_classes, target_names=CLASS_NAMES))

## 13. Per-Class Accuracy

In [ ]:
print(f"{'Class':<15} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print("-" * 45)
for i, name in enumerate(CLASS_NAMES):
    mask = y_test == i
    cls_correct = np.sum((y_test == pred_classes) & mask)
    cls_total = np.sum(mask)
    cls_acc = (cls_correct / cls_total * 100) if cls_total > 0 else 0
    print(f"  {name:<15}: {cls_correct:>4}/{cls_total:<4} ({cls_acc:>5.1f}%)")

## 14. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, pred_classes)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cbar_kws={'label': 'Count'}, ax=ax)
plt.title('Confusion Matrix — Vehicle Audio Classification', fontsize=14, pad=20)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('Actual Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix_audio.png')
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
print(f"Confusion matrix saved to: {cm_path}")
plt.show()

## 15. Sample Predictions

In [ ]:
# Show a few sample predictions
print("\nSample predictions (first 15):")
results_df.head(15)